В данном ноутбуке проходит сбор данных по выступлениям гонщиков в формуле 1 с 2018 по 2025 годы.
Данные собираются отдельно для каждого года из-за длительности скачиваний результов с официального портала формулы 1, а также особенностей обработки данных в зависимости от сезона.

#Загрузка данных с портала ф1

Данные загружаем с портала fast f1 - https://pypi.org/project/fastf1/.

In [ ]:
pip install fastf1

ERROR: Operation cancelled by user


Код ниже является весьма громоздким, и выполняется большое количество времени, что вызвано тем, что Google Collab имеет ограничение на количество запросов за условный промежуток времени, что вынуждает нас искусственно замедлять загрузку. После загрузки одного года, необходимо около часа для обнуления количества запросов.

In [ ]:
import fastf1 as f1
import pandas as pd
import time

f1.Cache.enable_cache('f1_cache')
request_count = 0
hour_start = time.time()

def check_limit():
    """Проверяет лимит запросов и ждет если нужно"""
    global request_count, hour_start
    request_count += 1
    print(f"Запросов в этом часу: {request_count}")
    elapsed = time.time() - hour_start # сбрасываем счётчки, если работаем больше часа
    if elapsed > 3600:
        request_count = 1
        hour_start = time.time()
        return

    if request_count >= 450:
        wait_time = 3600 - elapsed + 60  # Ждем до конца часа + 1 минута
        print(f"\n Лимит API: {request_count}/500 запросов")
        time.sleep(wait_time)
        request_count = 0
        hour_start = time.time()

def load_with_retry(season, event_name, session_type, max_retries=5):
    """Загружает сессию с повторными попытками при ошибке 500 calls"""
    for attempt in range(max_retries):
        try:
            check_limit()
            session = f1.get_session(season, event_name, session_type)
            session.load(telemetry=False, weather=False, messages=False)
            time.sleep(3)
            return session

        except Exception as e:
            error_msg = str(e)
            if "500 calls" in error_msg or "RateLimit" in error_msg:
                print(f" Лимит API при загрузке {session_type}!")
                if attempt < max_retries - 1:
                    # Ждем 5 минут и пробуем снова
                    wait_time = 300
                    print(f"Попытка {attempt + 2}/{max_retries} через {wait_time/60} минут...")
                    time.sleep(wait_time)
                else:
                    print(f"Все попытки исчерпаны, жду 1 час")
                    time.sleep(3600)
                    global request_count, hour_start
                    request_count = 0
                    hour_start = time.time()
            else:
                return None

    return None

def get_practice_positions(session):
    """Получает позиции из практики по лучшим кругам"""
    try:
        if session is None or session.laps.empty:
            return {}
        quick_laps = session.laps.pick_quicklaps()
        if quick_laps.empty:
            return {}
        best_laps = quick_laps.loc[quick_laps.groupby('Driver')['LapTime'].idxmin()]
        best_laps = best_laps.sort_values('LapTime')
        positions = {}
        for idx, (_, row) in enumerate(best_laps.iterrows()):
            positions[row['Driver']] = idx + 1
        return positions
    except Exception as e:
        print(f"    Ошибка обработки практики: {e}")
        return {}

def get_session_positions(session):
    """Получает позиции из квалификации или спринта"""
    try:
        if session is None or not hasattr(session, 'results') or session.results is None:
            return {}
        positions = {}
        for _, row in session.results.iterrows():
            if 'Position' in row and pd.notna(row['Position']):
                positions[row['Abbreviation']] = int(row['Position'])
        return positions
    except Exception as e:
        print(f"    Ошибка обработки сессии: {e}")
        return {}

def has_sprint(season, event_name):
    """Проверяет наличие спринта"""
    sprint_codes = ['Sprint', 'S', 'Sprint Qualifying', 'Sprint Shootout']
    for code in sprint_codes:
        print(f"    Проверка {code}")
        session = load_with_retry(season, event_name, code)
        if session is not None:
            print(f"Найден спринт: {code}")
            return True, session
    return False, None

def load_season(season):
    """Загружает один сезон"""
    all_races = []
    successful = 0
    failed = []
    try:
        check_limit()
        schedule = f1.get_event_schedule(season)
        print(f"Найдено событий: {len(schedule)}")
        time.sleep(5)
    except Exception as e:
        print(f"Ошибка загрузки календаря: {e}")
        return None

    total_events = len(schedule)
    for idx, (_, event) in enumerate(schedule.iterrows()):
        if idx == 0:  # Пропускаем тесты
            continue

        event_name = event['EventName']
        event_round = event['RoundNumber']
        event_location = event['Location']
        print(f"Гонка {idx}/{total_events-1}: {event_name} (Раунд {event_round})")
        race = load_with_retry(season, event_name, 'R')

        if race is None:
            print(f" Не удалось загрузить гонку, пропускаю...")
            failed.append(f"{event_name} (Race)")
            continue

        print(f"Загружено {len(race.results)} пилотов")
        df = pd.DataFrame({
            'Year': season,
            'Round': event_round,
            'Place': event_location,
            'Driver': race.results['Abbreviation'],
            'Team': race.results['TeamName'],
            'Result': range(1, len(race.results) + 1),
            'Start_Position': race.results['GridPosition'].fillna(0).astype(int)
        })

        quali = load_with_retry(season, event_name, 'Q')
        if quali is not None:
            quali_pos = get_session_positions(quali)
            df['Q'] = df['Driver'].map(quali_pos).fillna(0).astype(int)
        else:
            print(f"Нет квалификации")
            df['Q'] = 0

        sprint_exists, sprint_session = has_sprint(season, event_name)
        if sprint_exists:
            sprint_pos = get_session_positions(sprint_session)
            df['Sprint'] = df['Driver'].map(sprint_pos).fillna(0).astype(int)
            print(f"Загружено {len(sprint_pos)} пилотов")
        else:
            df['Sprint'] = 0

        for fp_num, fp_code in enumerate(['FP1', 'FP2', 'FP3'], 1):
            fp = load_with_retry(season, event_name, fp_code)
            if fp is not None:
                fp_pos = get_practice_positions(fp)
                df[f'P{fp_num}'] = df['Driver'].map(fp_pos).fillna(0).astype(int)
            else:
                df[f'P{fp_num}'] = 0
        all_races.append(df)
        successful += 1
        temp_df = pd.concat(all_races, ignore_index=True)
        temp_df.to_csv(f'f1_{season}_temp.csv', index=False)
        print(f"Промежуточный результат сохранен")

        if idx < total_events - 1:
            print(f"\nПауза в две минуты перед следующей гонкой")
            for i in range(120, 0, -1):
                time.sleep(1)

    if all_races:
        final_df = pd.concat(all_races, ignore_index=True)
        filename = f'f1_data_{season}_complete.csv'
        final_df.to_csv(filename, index=False)
        return final_df
    else:
        return None
while True:
    try:
        year = input("\nСезон: ").strip()

        if year.lower() in ['exit', 'quit', 'q', 'выход']:
            break

        year = int(year)
        if year < 2018 or year > 2025:
            print("Год должен быть от 2018 до 2025")
            continue

        global request_count, hour_start
        request_count = 0
        hour_start = time.time()
        result = load_season(year)

        cont = input("\nЗагрузить другой год?: ").strip().lower()
        if cont not in ['да', 'yes', 'y', 'д']:
            break

    except ValueError:
        print("Пожалуйста, введите число от 2018 до 2025")
    except KeyboardInterrupt:
        print("\n\nПрограмма прервана пользователем")
        break
    except Exception as e:
        print(f"Неожиданная ошибка: {e}")

print("\nПрограмма завершена!")

ModuleNotFoundError: No module named 'fastf1'

# Дополняем данные по годам

К сожалению, данные с fast f1 по запросу "race" - не содержат многих признаков, таких как возраст гонщика, финансовые возможности команды, длительности питстопов и т.д. Эти данные собраны из ресурсов интернета автором, и представлены в файлах json формата в папке f1_statistical_data. Также есть резервная копия с пояснениями - reserve_all_statistical_data.txt

In [ ]:
pip install fastf1

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import fastf1 as f1
os.makedirs('f1_data', exist_ok=True)
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
with open(f'f1_data/tracks.json', 'w', encoding='utf-8') as f:
        json.dump(tracks, f, indent=2, ensure_ascii=False)

In [ ]:
with open('f1_data/tracks.json', 'r') as f:
    all_tracks = json.load(f)

In [ ]:
with open('f1_data/year_of_birthday_of_drivers.json', 'r') as f:
    drivers_birthdays = json.load(f)

In [ ]:
def apply_tracks(data, tracks):
    data["CircleLength"] = data['Place'].apply(lambda x: tracks[x][0])
    data["Quantity_of_circles"] = data['Place'].apply(lambda x: tracks[x][1])
    data["Quantity_of_drs"] = data['Place'].apply(lambda x: tracks[x][2])
    data["Quantity_of_turns"] = data['Place'].apply(lambda x: tracks[x][3])
    data["Tire_wear"] = data['Place'].apply(lambda x: tracks[x][4])
    data["Speed_of_turns"] = data['Place'].apply(lambda x: tracks[x][5])
    data["Downforce"] = data['Place'].apply(lambda x: tracks[x][6])
    data["Brake_load"] = data['Place'].apply(lambda x: tracks[x][7])
    data["Streets_or_not"] = data['Place'].apply(lambda x: tracks[x][8])
    data["Speed_of_track"] = data['Place'].apply(lambda x: tracks[x][9])
    data["PitStop_losing_time"] = data['Place'].apply(lambda x: tracks[x][10])
    return data

Ниже осуществлён сбор данных о погодных данных из fast f1

In [ ]:
def apply_weather(data, year):
    data['AirTemp'] = None
    data['TrackTemp'] = None
    data['Humidity'] = None
    data['Rainfall'] = None
    data['WindSpeed'] = None
    data['Pressure'] = None

    schedule = f1.get_event_schedule(year)
    for idx, event in schedule.iterrows():
        if idx == 0:
            continue
        session = f1.get_session(year, event['Location'], 'R')
        session.load(telemetry=False, messages=False)
        weather = session.weather_data
        mask = data['Place'] == event['Location']
        data.loc[mask, 'AirTemp'] = weather['AirTemp'].median()
        data.loc[mask, 'TrackTemp'] = weather['TrackTemp'].mean()
        data.loc[mask, 'Humidity'] = weather['Humidity'].median()
        data.loc[mask, 'Rainfall'] = weather['Rainfall'].any()
        data.loc[mask, 'WindSpeed'] = weather['WindSpeed'].median()
        data.loc[mask, 'Pressure'] = weather['Pressure'].median()
    return data

In [ ]:
with open('f1_data/f1_salaries_by_code.json', 'r') as f:
    salaries = json.load(f)

In [ ]:
with open('f1_data/team_financial_weight.json', 'r') as f:
    engines = json.load(f)

In [ ]:
with open('f1_data/new_regulations.json', 'r') as f:
    new_regulations = json.load(f)

## 2018

In [ ]:
data_2018 = pd.read_csv("formula1_2018_fixed.csv")

In [ ]:
with open('f1_data/engines_manufacters_2018.json', 'r') as f:
    engines = json.load(f)
data_2018['Engine'] = data_2018['Team'].map(engines)

In [ ]:
data_2018 = apply_tracks(data_2018, all_tracks)

In [ ]:
data_2018["Age"] = data_2018['Driver'].apply(lambda x: (2018 - drivers_birthdays[x]))

In [ ]:
with open('f1_data/average_pit_stops_2018.json', 'r') as f:
    pit_stops = json.load(f)
data_2018['Average_pit_stop'] = data_2018['Team'].map(pit_stops)

In [ ]:
data_2018 = apply_weather(data_2018, 2018)

req         WARNING 	DEFAULT CACHE ENABLED! (24.0 KB) /root/.cache/fastf1
core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.1]
INFO:fastf1.fastf1.core:Loading data for Bahrain Grand Prix - Race [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
INFO:fastf1.api:Fetching driver list...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loadin

In [ ]:
with open('f1_data/race_results_2017.json', 'r') as f:
    race_results_2017 = json.load(f)

data_2018["Result_last_year"] = data_2018.apply(
    lambda row: race_results_2017.get('2017', {}).get(row['Place'], {}).get(row['Driver'], None),
    axis=1
)

In [ ]:
data_2018['Salary'] = data_2018.apply(
    lambda row: f1_salaries_by_code.get(2018, {}).get(row["Driver"]),
    axis=1
)

In [ ]:
data_2018['Team_financial_power'] = data_2018['Team'].map(team_financial_weight)

In [ ]:
del data_2018['Fine']

In [ ]:
data_2018['Start_Position'] = data_2018['Start_Position'].apply(
    lambda x: x if x != 0 else 0
)

In [ ]:
data_2018['Regulation_Impact'] = data_2018['Year'].astype(str).map(new_regulations)

In [ ]:
data_2018.to_csv('f1_data_2018_full.csv', index=False)

##Остальные года

In [ ]:
def load_new_season(year):
    filename = f'formula1_{year}_fixed.csv'
    data = pd.read_csv(filename)

    filename = f'f1_data/engines_manufacters_{year}.json'
    with open(filename, 'r') as f:
        engines = json.load(f)
    data['Engine'] = data['Team'].map(engines)
    data = apply_tracks(data, all_tracks)
    data["Age"] = data['Driver'].apply(lambda x: (year - drivers_birthdays[x]))
    filename = f'f1_data/average_pit_stops_{year}.json'
    with open(filename, 'r') as f:
        pit_stops = json.load(f)
    data['Average_pit_stop'] = data['Team'].map(pit_stops)
    data = apply_weather(data, year)
    filename = f'formula1_{year - 1}_fixed.csv'
    data_last_year = pd.read_csv(filename)
    data = data.merge(data_last_year[['Driver', 'Place', 'Result']], on = ['Driver', 'Place'],
                  how = 'left', suffixes=('',"_last_year"))
    data['Salary'] = data.apply(
    lambda row: f1_salaries_by_code.get(year, {}).get(row["Driver"]),
    axis=1
    )
    data['Team_financial_power'] = data['Team'].map(team_financial_weight)
    data['Start_Position'] = data['Start_Position'].apply(
    lambda x: x if x != 0 else 0
    )
    data['Regulation_Impact'] = data['Year'].astype(str).map(new_regulations)
    nan_count = data['Regulation_Impact'].isna().sum()
    if nan_count == len(data['Regulation_Impact']):
        data['Regulation_Impact'] = data['Year'].map(new_regulations)
    data = data.drop_duplicates()
    data.to_csv(f'f1_data_{year}_full.csv', index=False)

In [ ]:
load_new_season(2019)

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.1]
INFO:fastf1.fastf1.core:Loading data for Bahrain Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
INFO:fastf1.fastf1.req:Using cached data for session_info
req            INFO 	Using cached data for driver_info
INFO:fastf1.fastf1.req:Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
INFO:fastf1.fastf1.req:Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
INFO:fastf1.fastf1.req:Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
INFO:fastf1.fastf1.req:Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
INFO:fastf1.fastf1.req:Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
INFO:fastf1.fastf1.req:Using cached data for timing_app_data
core         

In [ ]:
load_new_season(2020)

req         WARNING 	DEFAULT CACHE ENABLED! (24.0 KB) /root/.cache/fastf1
core           INFO 	Loading data for Spanish Grand Prix - Race [v3.8.1]
INFO:fastf1.fastf1.core:Loading data for Spanish Grand Prix - Race [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
INFO:fastf1.api:Fetching driver list...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loadin

In [ ]:
load_new_season(2021)


core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.1]
INFO:fastf1.fastf1.core:Loading data for Bahrain Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
INFO:fastf1.fastf1.req:Using cached data for session_info
req            INFO 	Using cached data for driver_info
INFO:fastf1.fastf1.req:Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
INFO:fastf1.fastf1.req:Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
INFO:fastf1.fastf1.req:Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
INFO:fastf1.fastf1.req:Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
INFO:fastf1.fastf1.req:Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
INFO:fastf1.fastf1.req:Using cached data for timing_app_data
core         

In [ ]:
load_new_season(2022)

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.1]
INFO:fastf1.fastf1.core:Loading data for Bahrain Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
INFO:fastf1.fastf1.req:Using cached data for session_info
req            INFO 	Using cached data for driver_info
INFO:fastf1.fastf1.req:Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
INFO:fastf1.fastf1.req:Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
INFO:fastf1.fastf1.req:Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
INFO:fastf1.fastf1.req:Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
INFO:fastf1.fastf1.req:Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
INFO:fastf1.fastf1.req:Using cached data for timing_app_data
core         

In [ ]:
load_new_season(2023)

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.1]
INFO:fastf1.fastf1.core:Loading data for Bahrain Grand Prix - Race [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
INFO:fastf1.api:Fetching driver list...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_status_d

In [ ]:
load_new_season(2024)

In [ ]:
load_new_season(2025)

core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.1]
INFO:fastf1.fastf1.core:Loading data for Australian Grand Prix - Race [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
INFO:fastf1.api:Fetching driver list...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_st

In [ ]:
with open(f'f1_data/team_factory_status.json', 'r') as f:
        team_factory = json.load(f)


In [ ]:
with open(f'f1_data/teammate_strength_first_3_years.json', 'r') as f:
        teammate_strength = json.load(f)

In [ ]:
with open(f'f1_data/driver_nationalities.json', 'r') as f:
        driver_nationalities = json.load(f)

In [ ]:
data = pd.read_csv("f1_data_2020_full.csv")

In [ ]:
data['Regulation_Impact'] = data['Year'].map(new_regulations)
data = data.drop_duplicates()

In [ ]:
def weighted_early_season_form(df):
    """
    Раздельная обработка avg_last_3 и avg_season для первых гонок
    """
    df = df.copy()
    df = df.sort_values(['Year', 'Round', 'Driver'])

    for year in df['Year'].unique():
        for driver in df[df['Year'] == year]['Driver'].unique():
            mask = (df['Year'] == year) & (df['Driver'] == driver)
            races = df[mask].sort_values('Round')
            season_results = []

            for idx, row in races.iterrows():
                current_round = row['Round']
                prev_year_res = row['Result_last_year']
                if pd.notna(prev_year_res) and prev_year_res > 20:
                    prev_year_res = 20.0
                if len(season_results) >= 3:
                    df.loc[idx, 'avg_last_3'] = sum(season_results[-3:]) / 3
                    df.loc[idx, 'avg_season'] = sum(season_results) / len(season_results)

                elif pd.notna(prev_year_res):
                    if len(season_results) == 0:
                        df.loc[idx, 'avg_last_3'] = prev_year_res
                        df.loc[idx, 'avg_season'] = prev_year_res
                    elif len(season_results) == 1:
                        df.loc[idx, 'avg_last_3'] = (season_results[0] + prev_year_res) / 2
                        df.loc[idx, 'avg_season'] = (season_results[0] + prev_year_res) / 2
                    elif len(season_results) == 2:
                        df.loc[idx, 'avg_last_3'] = (sum(season_results) + prev_year_res) / 3
                        df.loc[idx, 'avg_season'] = (sum(season_results) + prev_year_res) / 3

                else:
                    if len(season_results) > 0:
                        df.loc[idx, 'avg_last_3'] = sum(season_results) / len(season_results)
                        df.loc[idx, 'avg_season'] = sum(season_results) / len(season_results)
                if pd.notna(row['Result']):
                    season_results.append(row['Result'])
    for col in ['avg_last_3', 'avg_season']:
        df[col] = df[col].fillna(df.groupby(['Year', 'Round'])[col].transform('mean'))
        df[col] = df[col].fillna(df[col].median())

    return df


def calculate_points_ratio(df):
    """
    Упрощённое отношение очков с напарником
    """
    df = df.copy()
    df['teammate_points'] = np.nan

    for (year, round_num, team), group in df.groupby(['Year', 'Round', 'Team']):
        if len(group) == 2:
            drivers = group['Driver'].tolist()
            pts = group.set_index('Driver')['Points'].to_dict()

            df.loc[(df['Year'] == year) & (df['Round'] == round_num) & (df['Driver'] == drivers[0]), 'teammate_points'] = pts[drivers[1]]
            df.loc[(df['Year'] == year) & (df['Round'] == round_num) & (df['Driver'] == drivers[1]), 'teammate_points'] = pts[drivers[0]]
    mask_both_zero = (df['Points'] == 0) & (df['teammate_points'] == 0)
    df['points_ratio'] = np.where(
        mask_both_zero,
        1.0,
        df['Points'] / (df['teammate_points'] + 1e-6)
    )
    df['points_ratio_log'] = np.log1p(df['points_ratio'])
    df['avg_points_ratio'] = df.groupby('Driver')['points_ratio'].transform(
        lambda x: x.shift(1).expanding().mean()
    )
    df['avg_points_ratio'] = df['avg_points_ratio'].fillna(1.0)

    return df

In [ ]:
for i in range(2018, 2026):
    filename = f'f1_data_{i}_full.csv'
    data = pd.read_csv(filename)
    data['Regulation_Impact'] = data['Year'].map(new_regulations)
    data = data.drop_duplicates()
    data['Factorys_or_not'] = data['Team'].map(team_factory)
    data['Strengh_of_first_partners'] = data['Driver'].map(teammate_strength)
    data['Driver_nationalitie'] = data['Driver'].map(driver_nationalities)

    data['Points'] = data['Result'].apply(
      lambda x: race_points[int(x)] if x <= 10 else 0
    )

    data['Points'] = data.apply(
       lambda row: row['Points'] + sprint_points.get(int(row['Sprint']), 0) if row['Sprint'] <= 8 else row['Points'],
      axis=1
    )

    data = data.sort_values(['Round', 'Result'])
    data['cumulative_points'] = data.groupby(['Driver'])['Points'].cumsum()
    data['Points_before_race'] = data.groupby(['Driver'])['cumulative_points'].shift(1)

    data['Championship_position_before'] = data.groupby(['Year', 'Round'])['Points_before_race'].rank(
       method='min', ascending=False
    )

    data['Points_before_race'] = data['Points_before_race'].fillna(0)
    data = calculate_points_ratio(data)
    data = weighted_early_season_form(data)
    del data['cumulative_points']
    del data['Points']
    data.to_csv(f'f1_data_{i}_full.csv', index=False)


В следующем ноутбуке файлы за все года будут собраны в один, подвергнутся дополнительной обработке. Также там будут пояснения о всех признаках, которые есть о данных.